# Hydranomic - Gestão Autônoma de Metas e Configurações

Notebook dedicado para recepção, gerenciamento, validação e persistência do estado de metas e configurações na **WEGnology**.
- Processa requisições de criação e atualização de metas (`consumo-mensal`, `reducao_mensal`, `gasto-maximo-mensal`, `economia-mensal`) preservando todos os tipos simultaneamente.
- Gerencia e persiste preferências de notificações gerais e separadas por administrador logado (`notificacoes_usuarios`).
- Gera os artefatos `config.json` e `progresso_meta.json` no `OUTPUT_DIR` para sincronização com os **Application Files** da WEGnology.

In [ ]:
import os
import sys
import json
import calendar
from pathlib import Path
from typing import Any, Dict, List, Optional
from datetime import datetime

# Garante inclusao do diretorio atual e pastas de modulos no sys.path
dir_atual = Path.cwd()
candidatos_path = [dir_atual, dir_atual / 'notebooks' / 'scripts', dir_atual / 'notebooks' / 'jupyter']
for p in candidatos_path:
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from wegnology_adapter import (
    obter_diretorio_input,
    obter_diretorio_output,
    carregar_json,
    salvar_json,
    carregar_contexto
)

from metas import (
    carregar_config,
    salvar_config,
    salvar_notificacoes,
    criar_meta,
    solicitar_exclusao_meta,
    confirmar_exclusao_meta,
    avaliar_progresso_meta
)

print('✓ Modulo de Metas e Configuracoes inicializado com sucesso!')
print('  - INPUT_DIR: ', obter_diretorio_input())
print('  - OUTPUT_DIR:', obter_diretorio_output())


---
## Bloco 2: Parâmetros Injetáveis (Papermill / WEGnology Workflow)
*(Esta célula pode ser sobrescrita automaticamente durante a execução agendada ou via webhook na WEGnology)*

In [ ]:
ESTADO = 'metas'
CONSUMO_ATUAL_M3 = None
CONSUMO_MEDIDO_M3 = None
PERSISTIR_ARQUIVOS = True


---
## Bloco 3: Leitura de Contexto WEGnology e Avaliação do Progresso de Metas
Processa payloads de `definir_meta` e `salvar_notificacoes` recebidos via `context.json`, preservando múltiplos tipos de metas e preferências por usuário.

In [ ]:
contexto_ativo = carregar_contexto()
print(f'Contexto WEGnology recebido: {len(contexto_ativo)} chaves encontradas.')
if contexto_ativo:
    acao_ctx = contexto_ativo.get('acao') or (contexto_ativo.get('body', {}).get('acao') if isinstance(contexto_ativo.get('body'), dict) else None)
    print(f' - Acao detectada no contexto: {acao_ctx}')

# Se o consumo nao for injetado via parametro, avaliar_progresso_meta busca de consumo_mes.json ou conta.json
consumo_avaliar = CONSUMO_MEDIDO_M3 if CONSUMO_MEDIDO_M3 is not None else CONSUMO_ATUAL_M3
relatorio_meta = avaliar_progresso_meta(consumo_atual_m3=consumo_avaliar, salvar=PERSISTIR_ARQUIVOS)

# Assegura a persistencia de config.json no OUTPUT_DIR
if PERSISTIR_ARQUIVOS:
    cfg_atual = carregar_config()
    salvar_config(cfg_atual)


---
## Bloco 4: Auditoria de Metas Ativas e Alertas Gerados

In [ ]:
cfg = carregar_config()
print('=' * 60)
print('              STATUS ATUAL DE METAS E CONFIGURACOES          ')
print('=' * 60)
print('1. Meta de Consumo Ativa (meta_ativa):')
print('   ', cfg.get('meta_ativa'))
print('\n2. Metas Ativas por Tipo (metas_ativas_por_tipo):')
for t, m in cfg.get('metas_ativas_por_tipo', {}).items():
    print(f"   - [{t}]: {m.get('nome')} | Valor: {m.get('valor')} {m.get('unidade')} | Vigencia: {m.get('periodo')}")
print('\n3. Configuracao de Notificacoes:')
print('   - Geral:', cfg.get('notificacoes'))
print('   - Por Usuario:', list(cfg.get('notificacoes_usuarios', {}).keys()))
print('\n4. Alertas Ativos:')
alertas = relatorio_meta.get('alertas_ativos', []) if isinstance(relatorio_meta, dict) else []
if alertas:
    for a in alertas:
        print(f"   ! [{a.get('severidade', 'info').upper()}] {a.get('titulo')}: {a.get('mensagem')}")
else:
    print('   Nenhum alerta ativo no momento.')
print('=' * 60)


---
## Bloco 5: Verificação dos Arquivos Persistidos no `OUTPUT_DIR`

In [ ]:
dir_out = obter_diretorio_output()
print(f'Arquivos no OUTPUT_DIR ({dir_out}):')
for arq in sorted(dir_out.glob('*.json')):
    print(f' - {arq.name} ({arq.stat().st_size} bytes)')
